# Data Ingestion
## Stripe API Documentation for Agentic Customer Support RAG

### Objectives
1. **Web scraping best practices** for technical documentation
2. **Multiple chunking strategies** comparison and evaluation
3. **Metadata extraction** for enhanced retrieval
4. **Data quality validation** and preparation
5. **Storage format optimization** for different vector databases

### Project overview
- Web scraper for Stripe documentation (API Reference, Guides, Code Examples)
- Implementation of 5 chunking strategies:
  1. Fixed-size with overlap
  2. Recursive character splitting
  3. Semantic chunking (LlamaIndex)
  4. Sentence-window retrieval
  5. Hierarchical parent-child chunking
- Comprehensive metadata schema
- Export formats for ChromaDB, Pinecone, and Weaviate

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, asdict
from pathlib import Path
import time
import json

# StripeDoc hash
import hashlib

# web scraping
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from tqdm.auto import tqdm


import pandas as pd

## Configuration & Data Models

In [ ]:
@dataclass
class StripeDoc:
    """Data model for scraped Stripe documentation"""
    url: str
    title: str
    content: str
    doc_type: str  # 'api_reference', 'guide', 'tutorial', 'code_example'
    category: str  # 'payments', 'billing', 'connect', etc.
    subcategory: Optional[str] = None
    language: Optional[str] = None  # For code examples: 'python', 'javascript', etc.
    difficulty: Optional[str] = None  # 'beginner', 'intermediate', 'advanced'
    code_blocks: List[str] = None
    scraped_at: str = None
    doc_id: str = None  # Hash-based unique identifier
    
    def __post_init__(self):
        if self.code_blocks is None:
            self.code_blocks = []
        if self.scraped_at is None:
            from datetime import datetime
            self.scraped_at = datetime.now().isoformat()
        if self.doc_id is None:
            self.doc_id = hashlib.md5(self.url.encode()).hexdigest()[:16]

@dataclass
class Chunk:
    """Data model for document chunks"""
    chunk_id: str
    doc_id: str
    content: str
    chunk_index: int
    chunking_strategy: str
    token_count: int
    char_count: int
    metadata: Dict
    parent_chunk_id: Optional[str] = None  # For hierarchical chunking
    child_chunk_ids: List[str] = None  # For hierarchical chunking
    
    def __post_init__(self):
        if self.child_chunk_ids is None:
            self.child_chunk_ids = []

# Configuration
CONFIG = {
    'base_url': 'https://docs.stripe.com',
    'output_dir': Path('./stripe_docs_data'),
    'raw_docs_file': 'raw_documents.json',
    'chunks_dir': 'chunks',
    'rate_limit_delay': 1.0,  # seconds between requests
    'max_docs': 100,  # Limit for demo purposes; set to None for full scrape
    'user_agent': 'RAG-Project',
    'timeout': 30,
    'encoding': 'cl100k_base',  # OpenAI's tiktoken encoding
}

# Create output directories
CONFIG['output_dir'].mkdir(exist_ok=True)
(CONFIG['output_dir'] / CONFIG['chunks_dir']).mkdir(exist_ok=True)

print("✓ Configuration loaded")
print(f"  Output directory: {CONFIG['output_dir']}")
print(f"  Max documents: {CONFIG['max_docs']}")

## Web Scraping

### Utilities

In [ ]:
class StripeDocScraper:
    """Responsible web scraper for Stripe documentation"""

    def __init__(self, config: Dict):
        self.config = config
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': config['user_agent']
        })
        self.visited_urls = set()
        self.docs = []

    def is_valid_stripe_doc_url(self, url: str) -> bool:
        """Check if URL is a valid Stripe documentation page"""
        parsed = urlparse(url)
        return (
                'docs.stripe.com' in url and
                not any(x in url for x in ['#', 'mailto:', '.pdf', '.zip']) and
                parsed.scheme in ['http', 'https']
        )

    def categorize_doc(self, url: str, soup: BeautifulSoup) -> Tuple[str, str, Optional[str]]:
        """Determine document type and category from URL and content"""
        url_lower = url.lower()

        # Determine doc_type
        if '/api/' in url_lower:
            doc_type = 'api_reference'
        elif '/get-started' in url_lower:
            doc_type = 'tutorial'
        else:
            doc_type = 'guide'

        # Determine category
        category = 'general'
        subcategory = None

        if '/payment-intents' in url_lower:
            category = 'payments'
            subcategory = 'payment_intents'
        elif '/charges' in url_lower:
            category = 'payments'
            subcategory = 'charges'
        elif '/billing' in url_lower or '/subscriptions' in url_lower:
            category = 'billing'
            if '/subscriptions' in url_lower:
                subcategory = 'subscriptions'
            elif '/invoices' in url_lower:
                subcategory = 'invoices'
        elif '/accounts' in url_lower:
            category = 'connect'
        elif '/issuing' in url_lower:
            category = 'issuing'
        elif '/terminal' in url_lower:
            category = 'terminal'
        elif '/identity' in url_lower:
            category = 'identity'
        elif '/customers' in url_lower:
            category = 'customers'

        return doc_type, category, subcategory

    def extract_code_blocks(self, soup: BeautifulSoup) -> List[str]:
        """Extract code blocks from documentation"""
        code_blocks = []
        for code in soup.find_all(['code', 'pre']):
            code_text = code.get_text().strip()
            if len(code_text) > 20:  # Ignore very short snippets
                code_blocks.append(code_text)
        return code_blocks

    def detect_language(self, soup: BeautifulSoup) -> Optional[str]:
        """Detect programming language from code examples"""
        for code_elem in soup.find_all(['code', 'pre']):
            classes = code_elem.get('class', [])
            for cls in classes:
                if 'python' in cls.lower():
                    return 'python'
                elif 'javascript' in cls.lower() or 'js' in cls.lower():
                    return 'javascript'
                elif 'ruby' in cls.lower():
                    return 'ruby'
                elif 'php' in cls.lower():
                    return 'php'
                elif 'curl' in cls.lower():
                    return 'curl'
        return None

    def clean_content(self, soup: BeautifulSoup) -> str:
        """Extract and clean main content from page"""
        # Remove script and style elements
        for script in soup(["script", "style", "nav", "header", "footer"]):
            script.decompose()

        # Try to find main content area
        main_content = soup.find(['main', 'article']) or soup.find('div', {'class': 'content'})
        if main_content:
            text = main_content.get_text(separator='\n', strip=True)
        else:
            text = soup.get_text(separator='\n', strip=True)

        # Clean up excessive whitespace
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        text = '\n'.join(lines)

        return text

    def scrape_page(self, url: str) -> Optional[StripeDoc]:
        """Scrape a single documentation page"""
        if url in self.visited_urls:
            return None

        self.visited_urls.add(url)

        try:
            time.sleep(self.config['rate_limit_delay'])
            response = self.session.get(url, timeout=self.config['timeout'])
            response.raise_for_status()

            soup = BeautifulSoup(response.content, 'html.parser')

            # Extract title
            title = soup.find('h1')
            title_text = title.get_text().strip() if title else urlparse(url).path.split('/')[-1]

            # Extract content
            content = self.clean_content(soup)

            # Skip if content is too short (likely not a real doc page)
            if len(content) < 100:
                return None

            # Categorize
            doc_type, category, subcategory = self.categorize_doc(url, soup)

            # Extract code blocks
            code_blocks = self.extract_code_blocks(soup)

            # Detect language
            language = self.detect_language(soup)

            # Create document
            doc = StripeDoc(
                url=url,
                title=title_text,
                content=content,
                doc_type=doc_type,
                category=category,
                subcategory=subcategory,
                language=language,
                code_blocks=code_blocks
            )

            return doc

        except Exception as e:
            print(f"Error scraping {url}: {e}")
            return None

    def discover_doc_urls(self, start_url: str, max_pages: int = 100) -> List[str]:
        """Discover documentation URLs from sitemap or by crawling"""
        urls = []
        to_visit = [start_url]
        visited = set()

        print("Discovering Stripe documentation URLs...")

        with tqdm(total=max_pages, desc="Discovering URLs") as pbar:
            while to_visit and len(urls) < max_pages:
                current_url = to_visit.pop(0)

                if current_url in visited or not self.is_valid_stripe_doc_url(current_url):
                    continue

                visited.add(current_url)
                urls.append(current_url)
                pbar.update(1)

                try:
                    time.sleep(self.config['rate_limit_delay'])
                    response = self.session.get(current_url, timeout=self.config['timeout'])
                    soup = BeautifulSoup(response.content, 'html.parser')

                    # Find all links
                    for link in soup.find_all('a', href=True):
                        href = link['href']
                        full_url = urljoin(current_url, href)

                        if (self.is_valid_stripe_doc_url(full_url) and
                                full_url not in visited and
                                full_url not in to_visit):
                            to_visit.append(full_url)

                except Exception as e:
                    print(f"Error discovering from {current_url}: {e}")
                    continue

        return urls

    def scrape_docs(self, urls: List[str]) -> List[StripeDoc]:
        """Scrape multiple documentation pages"""
        docs = []

        print(f"\nScraping {len(urls)} documentation pages...")

        for url in tqdm(urls, desc="Scraping pages"):
            doc = self.scrape_page(url)
            if doc:
                docs.append(doc)

        return docs


print("✓ Scraper utilities defined")

### Data Collection with Predefined URLs

In [ ]:
SEED_URLS = [
    # Payment Intents
    'https://docs.stripe.com/payments/payment-intents',
    'https://docs.stripe.com/payments/accept-a-payment',
    'https://docs.stripe.com/api/payment_intents',

    # Charges (legacy)
    'https://docs.stripe.com/api/charges',

    # Customers
    'https://docs.stripe.com/api/customers',
    'https://docs.stripe.com/payments/save-and-reuse',

    # Subscriptions & Billing
    'https://docs.stripe.com/billing/subscriptions/overview',
    'https://docs.stripe.com/api/subscriptions',
    'https://docs.stripe.com/api/invoices',
    'https://docs.stripe.com/billing/subscriptions/usage-based',

    # Payment Methods
    'https://docs.stripe.com/payments/payment-methods/overview',
    'https://docs.stripe.com/api/payment_methods',

    # Webhooks
    'https://docs.stripe.com/webhooks',
    'https://docs.stripe.com/api/events',

    # Refunds
    'https://docs.stripe.com/refunds',
    'https://docs.stripe.com/api/refunds',

    # Connect
    'https://docs.stripe.com/connect',
    'https://docs.stripe.com/api/accounts',

    # Error Handling
    'https://docs.stripe.com/error-handling',
    'https://docs.stripe.com/api/errors',
]

print(f"Using {len(SEED_URLS)} seed URLs for focused scraping")

In [ ]:
# Initialize scraper
scraper = StripeDocScraper(CONFIG)

# Option A: Use seed URLs and discover related pages
print("Starting with seed URLs and discovering related documentation...")
all_urls = scraper.discover_doc_urls(
    start_url=SEED_URLS[0], 
    max_pages=CONFIG['max_docs'] or 100
)

# Add our seed URLs to ensure they're included
all_urls = list(set(SEED_URLS + all_urls))

print(f"\nDiscovered {len(all_urls)} unique documentation URLs")

# Show sample URLs
print("\nSample URLs to scrape:")
for url in all_urls[:5]:
    print(f"  - {url}")
print("  ...")

In [ ]:
# Scrape the documents
stripe_docs = scraper.scrape_docs(all_urls[:CONFIG['max_docs']])

print(f"\n✓ Successfully scraped {len(stripe_docs)} documents")

# Save raw documents
output_file = CONFIG['output_dir'] / CONFIG['raw_docs_file']
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump([asdict(doc) for doc in stripe_docs], f, indent=2, ensure_ascii=False)

print(f"✓ Saved raw documents to {output_file}")

## Data Analysis & Quality Check

In [ ]:
# Load documents (if continuing from saved file)
# with open(CONFIG['output_dir'] / CONFIG['raw_docs_file'], 'r') as f:
#     stripe_docs_data = json.load(f)
#     stripe_docs = [StripeDoc(**doc) for doc in stripe_docs_data]

# Create DataFrame for analysis
df = pd.DataFrame([asdict(doc) for doc in stripe_docs])

print("Document Collection Statistics")
print("=" * 50)
print(f"Total documents: {len(df)}")
print(f"\nDocument types:")
print(df['doc_type'].value_counts())
print(f"\nCategories:")
print(df['category'].value_counts())
print(f"\nLanguages (for code examples):")
print(df['language'].value_counts())

# Content length analysis
df['content_length'] = df['content'].str.len()
df['num_code_blocks'] = df['code_blocks'].apply(len)

print(f"\nContent statistics:")
print(df['content_length'].describe())
print(f"\nCode blocks statistics:")
print(df['num_code_blocks'].describe())

In [ ]:
# Visualize distribution
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Document types
df['doc_type'].value_counts().plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Document Types Distribution')
axes[0, 0].set_xlabel('Document Type')
axes[0, 0].set_ylabel('Count')

# Categories
df['category'].value_counts().head(10).plot(kind='barh', ax=axes[0, 1], color='coral')
axes[0, 1].set_title('Top 10 Categories')
axes[0, 1].set_xlabel('Count')

# Content length distribution
df['content_length'].hist(bins=30, ax=axes[1, 0], color='lightgreen', edgecolor='black')
axes[1, 0].set_title('Content Length Distribution')
axes[1, 0].set_xlabel('Characters')
axes[1, 0].set_ylabel('Frequency')

# Code blocks distribution
df['num_code_blocks'].value_counts().sort_index().plot(kind='bar', ax=axes[1, 1], color='mediumpurple')
axes[1, 1].set_title('Code Blocks per Document')
axes[1, 1].set_xlabel('Number of Code Blocks')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(CONFIG['output_dir'] / 'data_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualizations saved")